In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, parent_dir)
from scripts.plotting import *

# ---------- Load vector field ----------
def load_vector_field(csv_path):
    df = pd.read_csv(csv_path)
    X = df[["x", "y"]].values
    V = df[["vx", "vy"]].values
    time = df["time"].values
    return X, V, time

# ---------- File organization ----------
path_map = {
    "straight_line": "./data/1d/straight_line.csv",
    "sine_curve": "./data/1d/sine_curve.csv",
    "branch_2": "./data/1d/branch_2.csv",
    "branch_4": "./data/1d/branch_4.csv",
    "rotation": "./data/2d/rotation.csv",
    "spiral": "./data/2d/spiral.csv",
    "saddle": "./data/2d/saddle.csv",
    "quadratic_source_sink": "./data/2d/quadratic_source_sink.csv"
}

# Custom layout
row1_names = ["straight_line", "sine_curve", "branch_2", "branch_4"]
row2_names = ["rotation", "spiral", "saddle", "quadratic_source_sink"]
plot_order = row1_names + row2_names

# ---------- 2×4 grid plot ----------
fig, axs = plt.subplots(2, 4, figsize=(20, 10))

for i, (ax, name) in enumerate(zip(axs.ravel(), plot_order)):
    X, V, time = load_vector_field(path_map[name])
    time = (time - np.min(time)) / (np.max(time) - np.min(time))  # normalize time globally

    tps_vf = ThinPlateSpline(X, n_control_points=100)
    tps_vf.fit(V, dof=15)

    if i == 0:
        stream_density = 0.4
        aspect = 2.5
    elif i == 1:
        stream_density = 0.8
        aspect = 2.0
    elif i < 4:
        stream_density = 0.8
        aspect = 1.5
    else:
        stream_density = 1
        aspect = "equal"

    plot_velocity_streamplot(
        X_2d=X,
        tps_vf=tps_vf,
        grid_density=1.0, 
        stream_density=stream_density,
        scatter_color=time,
        scatter_size=40,
        scatter_alpha=0.5,
        ax=ax,
        title=name.replace("_", " "),
        figsize=(5, 4),
        aspect=aspect,
        cmap="viridris",
        vmin=0.0,
        vmax=1.0,
        grid_size=50
    )

plt.tight_layout()
plt.show()

In [ ]:
# Ensure directory exists
save_dir = "./data/8_vf_collection"
os.makedirs(save_dir, exist_ok=True)

np.random.seed(42)
simulation_results = {}
noise = 0.2
extra_dim = 2

for i, (name, path) in enumerate(path_map.items()):
    # Load clean 2D data
    X_gt, V_gt, time = load_vector_field(path)

    # Add noise
    X_noisy = X_gt + np.random.normal(scale=noise, size=X_gt.shape)
    V_noisy = V_gt + np.random.normal(scale=noise, size=V_gt.shape)

    # Add dummy dimensions
    X_dummy = np.random.normal(scale=noise, size=(X_gt.shape[0], extra_dim))
    V_dummy = np.random.normal(scale=noise, size=(V_gt.shape[0], extra_dim))

    X = np.hstack([X_noisy, X_dummy])
    V = np.hstack([V_noisy, V_dummy])

    # Build dataframe
    df = pd.DataFrame(
        np.hstack([X, V, time[:, None]]),
        columns=[f"x{i+1}" for i in range(X.shape[1])]
              + [f"v{i+1}" for i in range(V.shape[1])]
              + ["true_time"]
    )

    # Save to CSV
    df.to_csv(os.path.join(save_dir, f"{name}.csv"), index=False)

    # Store in dict as well
    simulation_results[name] = {
        "X": X,
        "V": V,
        "true_time": time
    }